In [12]:
!pip install -q smolagents

In [76]:
from smolagents import (
    DuckDuckGoSearchTool,
    VisitWebpageTool,
)

## Dux Distributed Global Search

In [14]:
!pip install -q ddgs

In [15]:
from ddgs import DDGS

In [16]:
results = DDGS().text(
    "most popular programming language",
    max_results=5,
    safesearch="on",
    backend="google",
)
print(results)

[{'title': 'Most Popular Programming Languages - DevTopics', 'href': 'https://www.devtopics.com/most-popular-programming-languages/', 'body': 'Aug 3, 2014 · The topic is most popular programming languages, not which is the best programming language, which would be another discussion altogether. Bashing one or more …'}, {'title': '101 Great Computer Programming Quotes - DevTopics', 'href': 'https://www.devtopics.com/101-great-computer-programming-quotes/', 'body': 'Oct 13, 2016 · 210 Responses to “101 Great Computer Programming Quotes” purrl.net |** urls that purr **| Says: January 11th, 2008 at 4:04 pm This is one of the web’s most interesting stories …'}, {'title': 'Google Considered C# as the Native Language for Android', 'href': 'https://www.devtopics.com/category/c/', 'body': 'In general, I feel it’s better to select the right programming language for the job, rather than force the job to use my current favorite language. That said, Java fan Brian M. Clapper has written …'}, {'titl

In [17]:
# based on https://medium.com/@laurentkubaski/smolagents-duckduckgosearchtool-to-search-in-wikipedia-2578973bb131

class CustomDuckDuckGoSearchTool(DuckDuckGoSearchTool):
    name = "web_search"
    description = "Performs a web search for a query and returns a list of the top search results formatted as markdown with page titles and urls."
    inputs = {"query": {"type": "string", "description": "The search query to perform."}}
    output_type = "string"

    def __init__(self, max_results: int = 10, rate_limit: float | None = 1.0, backend: str = "auto", **kwargs):
        super().__init__(max_results=max_results, rate_limit=rate_limit, **kwargs)
        self.backend = backend # Add "backend" as new parameter

    def forward(self, query: str) -> str:
        self._enforce_rate_limit()
        results = self.ddgs.text(
            query=query,
            max_results=self.max_results,
            backend=self.backend) # there you go
        
        if len(results) == 0:
            raise Exception("No results found! Try a less restrictive/shorter query.")
        
        postprocessed_results = [
            f"{i+1}. [{result['title']}]({result['href']})\nBody: {result['body']}"
            for i, result in enumerate(results)
        ]

        return "## Search Results\n" + "\n".join(postprocessed_results)

In [18]:
tool = CustomDuckDuckGoSearchTool(
    max_results=5,
    rate_limit=1.0,
    backend="google")

result = tool(query='most popular programming language')
print(result)

## Search Results
1. [TIOBE index - Wikipedia](https://en.wikipedia.org/wiki/TIOBE_index)
Body: The TIOBE programming community index is a measure of popularity of programming languages , created and maintained by TIOBE Software BV, based in Eindhoven, the Netherlands. TIOBE stands for The Importance of Being Earnest, the title of an 1895 comedy...
2. [The Most Popular Programming Languages in Today's Tech World](https://www.linkedin.com/pulse/most-popular-programming-languages-todays-tech-world-tekvaly-8v50f)
Body: Moreover, the popularity of a programming language often determines its relevance and demand in the job market. Here, we explore some of the most popular programming languages , offering a comprehensive overview of their significance, usage statistics...
3. [Top Programming Languages 2024 - IEEE Spectrum](https://spectrum.ieee.org/top-programming-languages-2024)
Body: Stephen Cass is the special projects editor at IEEE Spectrum. Welcome to IEEE Spectrum ’s 11th annual ranki

## Visit web page

In [19]:
!pip install -q markdownify

In [20]:
visit_web_tool = VisitWebpageTool()

result = visit_web_tool.forward("https://www.zdnet.com/article/the-most-popular-programming-languages-in-2024-and-what-that-even-means/")
print(result)

The most popular programming languages in 2025 (and what that even means) | ZDNET

X

Trending  

* [Here's how to save like a shopping editor for Black Friday 2025](/home-and-office/black-friday-2025-everything-you-need-to-know-about-holiday-shopping/)
* [I'm tracking over 40 of my favorite Black Friday discounts live now](/article/best-early-black-friday-deals-2025/)
* [9 iPad sales out already for Black Friday](/article/best-early-black-friday-ipad-deals-2025/)
* [Shop these top deals on fitness trackers & smartwatches for Black Friday 2025](/article/best-black-friday-smartwatch-deals-2025/)
* [My 10 favorite tablet discounts live now for Black Friday 2025](/article/best-early-black-friday-tablet-deals-2025/)
* [Shop the best Black Friday Laptop 2025 discounts live now](/article/best-early-black-friday-laptop-deals-2025/)
* [Shop the best early Costco deals for Black Friday 2025](/home-and-office/best-early-black-friday-costco-deals-2025/)

* [Black Friday Walmart deals 2025: My top

In [24]:
!pip install -q readability-lxml

In [77]:
class CleanVisitWebpageTool(VisitWebpageTool):
    """
    Improved version of VisitWebpageTool that extracts and cleans the main content
    of a webpage for efficient LLM consumption.
    """

    name = "clean_visit_webpage"
    description = (
        "Visits a webpage at the given url and reads its content as a markdown string. Use this to browse webpages."
    )
    inputs = {
        "url": {
            "type": "string",
            "description": "The url of the webpage to visit.",
        }
    }
    output_type = "string"

    def __init__(self, max_output_length: int = 4000, clean_level: int = 2):
        super().__init__(max_output_length=max_output_length)
        self.clean_level = clean_level  # 0 = raw, 1 = cleaned, 2 = aggressively cleaned

    def _extract_main_content(self, html: str) -> tuple[str, str]:
        """
        Extracts the main content (article body) from the HTML using readability-lxml.
        Returns (title, main_html)
        """
        try:
            from readability.readability import Document
            doc = Document(html)
            title = doc.title() or ""
            main_html = doc.summary() or html
            return title, main_html
        except Exception:
            # Fallback: return raw HTML if readability fails
            return "", html

    def _clean_markdown(self, md: str) -> str:
        """
        Removes common noise, legal text, and repeated fluff from markdown text.
        """
        import re

        # Remove excessive blank lines
        md = re.sub(r'\n{2,}', '\n', md)

        junk_patterns = [
            r"(?i)(privacy policy|terms of use|cookie|subscribe|©|copyright).*",
            r"^\s*Share\s*$",
            r"(?i)related (articles|posts)",
            r"(?i)follow us",
            r"(?i)back to top",
        ]
        for pattern in junk_patterns:
            md = re.sub(pattern, '', md)

        # Remove short or generic markdown links (like [Home](...))
        md = re.sub(r'\[([^\]]{1,15})\]\([^)]+\)', r'\1', md)

        # Collapse multiple spaces and trim
        md = re.sub(r'[ \t]{2,}', ' ', md)
        return md.strip()

    def forward(self, url: str) -> str:
        try:
            import re
            import requests
            from markdownify import markdownify
            from requests.exceptions import RequestException
        except ImportError as e:
            raise ImportError(
                "You must install `markdownify`, `readability-lxml`, and `requests` "
                "to run CleanVisitWebpageTool: "
                "`pip install markdownify readability-lxml requests`."
            ) from e

        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()

            html = response.text
            title, main_html = self._extract_main_content(html)

            markdown_content = markdownify(main_html, heading_style="ATX").strip()

            # Apply cleaning if requested
            if self.clean_level >= 1:
                markdown_content = self._clean_markdown(markdown_content)

            # Add title as a header
            if title:
                markdown_content = f"# {title}\n{markdown_content}"

            # Truncate if too long
            markdown_content = self._truncate_content(markdown_content, self.max_output_length)

            return markdown_content

        except requests.exceptions.Timeout:
            return "The request timed out. Please try again later or check the URL."
        except RequestException as e:
            return f"Error fetching the webpage: {str(e)}"
        except Exception as e:
            return f"An unexpected error occurred: {str(e)}"

In [78]:
clean_visit_tool = CleanVisitWebpageTool()

result = clean_visit_tool.forward("https://www.zdnet.com/article/the-most-popular-programming-languages-in-2024-and-what-that-even-means/")
print(result)

# The most popular programming languages in 2025 (and what that even means) | ZDNET
photo\_Pawel/Getty Images
We ran [a piece last year summarizing an IEEE study of programming-language popularity](https://www.zdnet.com/article/want-a-programming-job-make-sure-you-learn-these-three-languages/) based on job listings. This article fostered conversation, including debates about whether the languages IEEE used in its survey were even languages.
Most of us are familiar with polls and poll results, especially during campaign seasons. Unfortunately, polls have long been proven to be far from accurate. Some polls have a natural bias for one party or the other (not for nefarious reasons, but just based on how they gather their data). Other polls have demographic or psychographic bias. The bottom line is simple: just because the numbers go up in one poll doesn't mean your candidate will win.
**Also: [Brace yourself: The era of 'citizen developers' creating apps is here, thanks to AI](https://www